In [4]:
!pip -q install pypdf faiss-cpu langchain_community langchain_text_splitters langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [5]:
import os
import requests
from pathlib import Path
from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI


In [8]:
DATA_DIR = Path("data")
INDEX_DIR = Path("faiss_index")

DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

PDF_URLS = [
    "https://arxiv.org/pdf/1706.03762.pdf",
    "https://arxiv.org/pdf/1810.04805.pdf",
    "https://arxiv.org/pdf/2005.11401.pdf"
]

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "google/gemini-2.5-flash"

TOP_K = 5

### Download PDF files

In [9]:
def download_file(url, save_path):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    
    with open(save_path, "wb") as f:
        f.write(response.content)


for url in PDF_URLS:
    file_name = url.split("/")[-1]
    save_path = DATA_DIR / file_name
    
    if not save_path.exists():
        download_file(url, save_path)

sorted([p.name for p in DATA_DIR.glob("*.pdf")])

['1706.03762.pdf', '1810.04805.pdf', '2005.11401.pdf']

### Load PDF documents

In [10]:
all_docs = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    
    for doc in docs:
        doc.metadata["file_name"] = pdf_path.name
        doc.metadata["page_number"] = int(doc.metadata.get("page", 0)) + 1
    all_docs.extend(docs)
    
len(all_docs), all_docs[0].metadata, all_docs[0].page_content[:100]

(50,
 {'producer': 'pdfTeX-1.40.25',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2024-04-10T21:11:43+00:00',
  'author': '',
  'keywords': '',
  'moddate': '2024-04-10T21:11:43+00:00',
  'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
  'subject': '',
  'title': '',
  'trapped': '/False',
  'source': 'data/1706.03762.pdf',
  'total_pages': 15,
  'page': 0,
  'page_label': '1',
  'file_name': '1706.03762.pdf',
  'page_number': 1},
 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and')